In [ ]:
import os
import gc
import cv2
import time
import random
import glob
from PIL import Image
import  matplotlib.pyplot as plt

# For data manipulation
import numpy as np
import pandas as pd

# Pytorch Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torch.cuda import amp

# Utils
from tqdm.auto import tqdm

# For Image Models
import timm

# Albumentations for augmentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

import warnings
warnings.filterwarnings("ignore")

## using gpu:1
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

def seed_everything(seed=123):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()

In [ ]:
class Customize_Model(nn.Module):
    def __init__(self, model_name, cls):
        super().__init__()
    def forward(self, image):
        x = self.model(image)
        x = self.linear(x)
        x = torch.expm1(x) if not self.training else x
        return x


class dinov2(nn.Module):
    def __init__(self, cls):
        super().__init__()
        self.cls_num = cls
        self.model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        feature_dim = self.model.embed_dim * 4  # 因為 cat 了 4 層
        self.heads = nn.ModuleList([
            nn.Linear(feature_dim, 1) for _ in range(self.cls_num)
        ])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        
    def forward(self, image):
        layers = self.model.get_intermediate_layers(image, n=4, reshape=True)
        pooled_layers = [self.pool(ly) for ly in layers]
        x = torch.cat(pooled_layers, dim=1).squeeze(-1).squeeze(-1) 
        out = torch.cat([head(x) for head in self.heads], dim=1)
        out = torch.expm1(out) if not self.training else out
        return out
model = dinov2(cls=5)

In [ ]:
def get_test_transform(img_size):
    return A.Compose([
        A.SmallestMaxSize(max_size=img_size, interpolation=3, p=1),
        # A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(p=1.0),
    ])

class Customize_Dataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df
        self.transforms = transforms
    
    def __getitem__(self, index):
        data = self.df.loc[index]
        img = cv2.imread(data['image_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if self.transforms:
            img = self.transforms(image=img)["image"]
            
        return {
            'image': torch.tensor(img, dtype=torch.float32),
        }
    
    def __len__(self):
        return len(self.df)

# CFG

In [ ]:
CFG= {
    'fold': 0,
    'img_size': 896,
    'TTA': 1,
    
    # 'model': './train_model/cv0_best.pth',
    'model': './train_model/cv0_soup.ts',
    # 'model': './test_model/dinov3/cv4_soup.ts',
}
CFG['model']= [ torch.load(CFG['model'], weights_only=False, map_location= 'cuda:0') ]
print(f"length of model: {len(CFG['model'])}")

# Prepare Dataset

In [ ]:
ch_df = df= pd.read_csv('../Data/CH_DATA1.csv')[['image_path', 'kfold']]
for i in range(len(ch_df)): ch_df.loc[i, 'name'] = ch_df.loc[i, 'image_path'].split('/')[-1]
df= pd.read_csv('../Data/train_sgkf.csv')
for i in range(len(df)):
    name_ = df.loc[i, 'image_path'].split('/')[-1]
    fold = ch_df[ch_df['name']==name_]['kfold'].values[0]
    df.loc[i, 'fold'] = fold


In [ ]:
# df= pd.read_csv('../Data/train_sgkf.csv')
train_df= df[df['fold']!=CFG['fold']].reset_index(drop=True)
valid_df= df[df['fold']==CFG['fold']].reset_index(drop=True)
print(f'train dataset: {len(train_df)}')
print(f'valid dataset: {len(valid_df)}')

valid_dataset= Customize_Dataset(valid_df, get_test_transform(CFG['img_size']))
valid_loader= DataLoader(valid_dataset, batch_size=1, shuffle=False, num_workers=0)
valid_df.head()

In [ ]:
# valid_df = pd.DataFrame()
# valid_df.loc[0, 'image_path'] = '../Data/test/ID1001187975.jpg'
# valid_dataset= Customize_Dataset(valid_df, get_test_transform(CFG['img_size']))
# valid_loader= DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=0)
# valid_df

In [ ]:
def inference(model, img):
    
    img= torch.unsqueeze(img, 0).cuda()
    for i, m in enumerate(model):
        with torch.no_grad():
            m.eval()
            
            imgs= torch.cat([img, img.flip(-1), img.flip(-2), img.flip(-1).flip(-2)], dim=0)
            pred= m(imgs[:CFG['TTA']])
            pred= pred.mean(dim=0)
                
        if i==0: preds= pred
        else: preds+= pred
            
    pred= preds/len(model)
    pred= pred.cpu().numpy()
    return pred

In [ ]:
count= 0
for i, data in enumerate(tqdm(valid_loader)):
    for j in range(len(data['image'])):
        img= data['image'][j]
        pred= inference(CFG['model'], img)
        valid_df.loc[count, 'pred_prob']= str(pred.tolist())
        count+= 1
valid_df.head()

In [ ]:
import ast
from metrics import *

all_label = valid_df.iloc[:,4:9].values
all_pred = np.vstack(valid_df["pred_prob"].apply(ast.literal_eval).apply(lambda x: np.array(x)))

# all_pred = all_pred[..., :3]
# Dry_Clover_g = all_pred[:, 2:3] - all_pred[:, 0:1]
# Dry_Dead_g = all_pred[:, 1:2] - all_pred[:, 2:3]
# Dry_Total_g = all_pred[:, 0:1] + all_pred[:, 1:2] + all_pred[:, 2:3]
# GDM_g = all_pred[:, 0:1] + all_pred[:, 2:3]
# all_pred = np.concatenate([all_pred, Dry_Total_g, GDM_g], axis=1)

r2 = weighted_r2_csiro(all_label, all_pred)
r2

In [ ]:
plt.figure(figsize=(20,5))
for i in range(5):
    plt.subplot(1,5,i+1)
    plt.hist(all_label[...,i], bins=50)
    plt.xlim(0, all_label[...,i].max()+10)
plt.show()

plt.figure(figsize=(20,5))
for i in range(5):
    plt.subplot(1,5,i+1)
    plt.hist(all_pred[...,i], bins=50)
    plt.xlim(0, all_label[...,i].max()+10)
plt.show()